In [11]:
from pathlib import Path
import ipynbname

REPO_ROOT = Path(ipynbname.path()).parent.parent
DATA_ROOT = REPO_ROOT / "data"

import pandas as pd
import geopandas as gpd


In [12]:
budynki = gpd.read_file(DATA_ROOT / "raw" / "floorspace" / "budynki_bdot10k.gpkg")
#lokale = gpd.read_file(DATA_ROOT / "raw" / "floorspace" / "lokale.gpkg")
dzialki = gpd.read_file(DATA_ROOT / "raw" / "floorspace" / "dzialki.gpkg")
budynki.head()

KeyboardInterrupt: 

In [ ]:
dzialki.head(30)

NameError: name 'dzialki' is not defined

In [5]:
# data column = dok_data -> transform to datetime (orignal format e.g.: 2022-10-24 02:00:00+02)
# price column = tran_cena_brutto -> transform to float
# -> remove rows where price is 0 or np.nan

budynki['tran_cena_brutto'] = budynki['tran_cena_brutto'].astype(float)
budynki = budynki[budynki['tran_cena_brutto'] > 0]
budynki = budynki[budynki['tran_cena_brutto'].notna()]

lokale['tran_cena_brutto'] = lokale['tran_cena_brutto'].astype(float)
lokale = lokale[lokale['tran_cena_brutto'] > 0]
lokale = lokale[lokale['tran_cena_brutto'].notna()]

dzialki['nier_cena_brutto'] = dzialki['nier_cena_brutto'].astype(float)
dzialki = dzialki[dzialki['nier_cena_brutto'] > 0]
dzialki = dzialki[dzialki['nier_cena_brutto'].notna()]

# take first 10 characters from dok_data and transform to datetime
budynki['dok_data'] = pd.to_datetime(budynki['dok_data'].str[:10])
lokale['dok_data'] = pd.to_datetime(lokale['dok_data'].str[:10])
dzialki['dok_data'] = pd.to_datetime(dzialki['dok_data'].str[:10])

# and transfort YYYY-MM-DD to datetime
budynki['dok_data'] = pd.to_datetime(budynki['dok_data'], format='%Y-%m-%d')
lokale['dok_data'] = pd.to_datetime(lokale['dok_data'], format='%Y-%m-%d')
dzialki['dok_data'] = pd.to_datetime(dzialki['dok_data'], format='%Y-%m-%d')




In [6]:
print(budynki['nier_rodzaj'].value_counts())
print(lokale['nier_rodzaj'].value_counts())
print(dzialki['nier_rodzaj'].value_counts())


nier_rodzaj
nieruchomoscLokalowa                 4677235
nieruchomoscGruntowaZabudowana       1383934
nieruchomoscBudynkowa                  35463
nieruchomoscGruntowaNiezabudowana      30991
Name: count, dtype: int64
nier_rodzaj
nieruchomoscLokalowa                 2562479
nieruchomoscGruntowaZabudowana         10232
nieruchomoscGruntowaNiezabudowana        701
nieruchomoscBudynkowa                    606
Name: count, dtype: int64
nier_rodzaj
nieruchomoscLokalowa                 2904783
nieruchomoscGruntowaNiezabudowana    1604539
nieruchomoscGruntowaZabudowana        551390
nieruchomoscBudynkowa                  13091
Name: count, dtype: int64


In [8]:
gminy = gpd.read_file(DATA_ROOT / "raw" / "shapefiles" / "PRG_jednostki_administracyjne_2021" / "A03_Granice_gmin.shp")
gminy['teryt'] = gminy['JPT_KOD_JE'].astype(str)

short_gminy = gminy[['teryt', 'JPT_NAZWA_', 'geometry']]

# Spatial join budynki with short_gminy
budynki = gpd.sjoin(budynki.to_crs(short_gminy.crs), short_gminy, how='left', predicate='within')
lokale = gpd.sjoin(lokale.to_crs(short_gminy.crs), short_gminy, how='left', predicate='within')
dzialki = gpd.sjoin(dzialki.to_crs(short_gminy.crs), short_gminy, how='left', predicate='within')










In [14]:
bud_teryt = pd.DataFrame(budynki['teryt_right'].value_counts())
lok_teryt = pd.DataFrame(lokale['teryt_right'].value_counts())
dz_teryt = pd.DataFrame(dzialki['teryt_right'].value_counts())

bud_teryt.columns = ['budynki']
lok_teryt.columns = ['lokale']
dz_teryt.columns = ['dzialki']




In [16]:
dz_teryt

,dzialki
teryt_right,
1465011,671326
2261011,311807
2061011,276282
0663011,190636
1261011,156096
...,...
2014022,1
0405022,1
2213021,1


In [ ]:
budynki['teryt'].sort_values().unique()